# 🔋Predicting Battery Degradation Through Cycling Data🔋

## Contributions
This is a solo project done by Lixin Nie


## Introduction

Lithium-ion batteries are widely used in modern applications, including consumer electronics, electric vehicles, and energy storage systems. A key challenge in their usage is performance degradation over time, which affects capacity, efficiency, and overall lifespan.

In this project, we analyze battery cycling data to better understand charge and discharge behavior and prepare the data for future degradation modeling. The dataset used comes from experimental battery tests and includes measurements such as voltage, current, temperature, and cycle-related information over time.

The goal of this notebook is to:
- Load and inspect the raw battery dataset
- Clean and preprocess the data for analysis
- Identify key features such as charge and discharge phases
- Prepare the dataset for further analysis, including cycle extraction and degradation modeling

By structuring and cleaning the data properly, we enable more meaningful analysis and modeling in later stages of the analysis.

# Data Collection

The data collection stage of the Data Science lifecycle involves gathering relevant data that will help answer our research questions. For this project, we used battery cycling data provided by the **Center for Advanced Life Cycle Engineering (CALCE)** at the University of Maryland, available at [https://calce.umd.edu/battery-data](https://calce.umd.edu/battery-data). CALCE provides open-access experimental data on lithium-ion battery performance, including continuous full and partial cycling, storage conditions, dynamic driving profiles, open-circuit voltage measurements, and impedance data. Battery form factors in the dataset include cylindrical, pouch, and prismatic cells.

The dataset includes high-resolution measurements such as voltage, current, capacity, temperature, and cycle count. These variables allow us to track how battery performance evolves over time and under repeated usage, which is critical for analyzing degradation patterns.

Since the data is collected and shared by a reputable research institution, it is considered reliable and suitable for scientific analysis. For this project, we selected a subset of the available cycling datasets that contain sufficient observations and relevant features for studying battery degradation. These datasets were then downloaded and prepared for further processing in the next stage of the data science lifecycle.

We are going to utulize Python modules such as **numpy**, **pandas**, and **matplotlib** were used to process, analyze, and visualize the data.

In [73]:
#importing the necessary packages
import numpy as np 
import pandas as pd
import matplotlib as plt


The dataset is stored in a `.txt` file format. Unlike standard CSV files, this dataset uses a tab (`\t`) as the delimiter. Therefore, we must explicitly specify the separator when loading the data using pandas.

We load the dataset into a pandas DataFrame for further processing and analysis.

In [74]:
#We would need additional argument "sep="\t"" since the seperator is not comma in this case. 
df = pd.read_csv("./CS2_8/CS2_8_1_19_10.txt", sep="\t")

#------------- Generate Informations ------------------#

print("Shape of dataframe (rows, columns):")
print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nDataframe info:")
print(df.info())
print("\nPreview of dataframe:")
df.head()

Shape of dataframe (rows, columns):
(3268, 30)

Column names:
['Time', 'Status code', 'Status category', 'Status color', 'Pgm code', 'Pgm step', 'Pgm para', 'Pgm cycle', 'mV', 'mA', 'Temperature', 'Duration', 'Charge count', 'Discharge count', 'Capacity', 'Analog input 1', 'Analog input 2', 'Analog input 3', 'Analog input 4', 'Digital input 1', 'Digital input 2', 'Digital input 3', 'Digital input 4', 'Digital output 1', 'Digital output 2', 'Digital output 3', 'Digital output 4', 'Analog output 1', 'Analog output 2', 'Unnamed: 29']

Dataframe info:
<class 'pandas.DataFrame'>
RangeIndex: 3268 entries, 0 to 3267
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Time              3268 non-null   float64
 1   Status code       3268 non-null   int64  
 2   Status category   3268 non-null   int64  
 3   Status color      3268 non-null   int64  
 4   Pgm code          3268 non-null   int64  
 5   Pgm step        

,Time,Status code,Status category,Status color,Pgm code,Pgm step,Pgm para,Pgm cycle,mV,mA,...,Digital input 2,Digital input 3,Digital input 4,Digital output 1,Digital output 2,Digital output 3,Digital output 4,Analog output 1,Analog output 2,Unnamed: 29
0,0.000000,8,3,3,0,1,2,2,4005,552,...,0,0,0,0,0,0,0,0,0,NaN
1,0.750233,8,3,3,0,1,2,2,4030,550,...,0,0,0,0,0,0,0,0,0,NaN
2,1.745283,8,3,3,0,1,2,2,4042,548,...,0,0,0,0,0,0,0,0,0,NaN
3,2.761917,8,3,3,0,1,2,2,4047,549,...,0,0,0,0,0,0,0,0,0,NaN
4,3.758400,8,3,3,0,1,2,2,4051,550,...,0,0,0,0,0,0,0,0,0,NaN


Proper data cleaning is a critical step in the data science pipeline. Even when datasets appear structured, unnecessary columns, inconsistent naming, and unclear units can hinder analysis and lead to errors.

For this specific dataframe, we are going to drop then columns are not useful for our analysis.

In [75]:
#Dropping the useless null columns
df = df.drop(columns=['Unnamed: 29'])

#Theres quite a bit of the all zero columns, we are going to use loops to identify them.
io_cols = [c for c in df.columns if any(
    c.startswith(prefix) for prefix in
    ['Analog input', 'Analog output', 'Digital input', 'Digital output']
)]

zero_io_cols = [c for c in io_cols if (df[c] == 0).all()]
df = df.drop(columns=zero_io_cols)

#Inspect cleaned df 
print("Shape of dataframe (rows, columns):")
print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nPreview of dataframe:")

df.head()


Shape of dataframe (rows, columns):
(3268, 15)

Column names:
['Time', 'Status code', 'Status category', 'Status color', 'Pgm code', 'Pgm step', 'Pgm para', 'Pgm cycle', 'mV', 'mA', 'Temperature', 'Duration', 'Charge count', 'Discharge count', 'Capacity']

Preview of dataframe:


,Time,Status code,Status category,Status color,Pgm code,Pgm step,Pgm para,Pgm cycle,mV,mA,Temperature,Duration,Charge count,Discharge count,Capacity
0,0.000000,8,3,3,0,1,2,2,4005,552,19,2,1,0,0
1,0.750233,8,3,3,0,1,2,2,4030,550,20,47,1,0,0
2,1.745283,8,3,3,0,1,2,2,4042,548,19,107,1,0,0
3,2.761917,8,3,3,0,1,2,2,4047,549,20,168,1,0,0
4,3.758400,8,3,3,0,1,2,2,4051,550,20,228,1,0,0


To make the data easier to work with, we will convert all numerical values to standard units and update the column names to clearly show the units of each measurement.

In [76]:
#Convertin time to seconds
df['Time'] = df['Time'] * 60

df = df.rename(columns={
    'Time':     'Time_s',
    'Duration': 'Duration_s'
})

#Scale the electrical units
df['mV'] = df['mV'] / 1000
df['mA'] = df['mA'] / 1000

df = df.rename(columns={
    'mV': 'Voltage_V',
    'mA': 'Current_A'
})

df.head()

,Time_s,Status code,Status category,Status color,Pgm code,Pgm step,Pgm para,Pgm cycle,Voltage_V,Current_A,Temperature,Duration_s,Charge count,Discharge count,Capacity
0,0.00000,8,3,3,0,1,2,2,4.005,0.552,19,2,1,0,0
1,45.01398,8,3,3,0,1,2,2,4.030,0.550,20,47,1,0,0
2,104.71698,8,3,3,0,1,2,2,4.042,0.548,19,107,1,0,0
3,165.71502,8,3,3,0,1,2,2,4.047,0.549,20,168,1,0,0
4,225.50400,8,3,3,0,1,2,2,4.051,0.550,20,228,1,0,0
